# Diagnóstico y limpieza del dataset integrado de empleos

Proyecto: **MineríaProtect** — Sesión 3 (Minería de Datos).

Este cuaderno trabaja **exclusivamente** sobre el dataset integrado denominado `empleos`.
La integración de las fuentes (JobHop v2 + taxonomía ESCO) **ya se realizó en una etapa
anterior** y quedó versionada; esta tarea NO reintegra nada.

El cuaderno se construyó **sin ejecutar**: el código está listo para que el equipo lo corra
posteriormente (Kernel → Restart & Run All). Ningún resultado impreso aquí es un resultado
de este cuaderno: los valores de referencia citados provienen de la auditoría verificada de la
rama (`md/AUDITORIA_RAMA_PRUEBA.md`) y sirven solo para interpretar lo que se espera observar.


## 1. Objetivo

- **Qué se analiza:** `empleos`, el dataset integrado de experiencias laborales (1 fila = 1 empleo
  de una persona, enriquecido con la ocupación y el área ocupacional).
- **De dónde proviene:** de la integración de las fuentes del proyecto — JobHop v2 (trayectorias,
  quién/cuándo/qué ocupación/nivel educativo) y ESCO (traducción de los códigos de ocupación a
  nombre y grupo ISCO-08).
- **Objetivo de la limpieza:** dejar un dataset **diagnosticado, verificado y documentado**: sin
  duplicados reales, con banderas explícitas para valores ausentes/desconocidos, categorías
  consistentes y fechas coherentes; listo para la minería de trayectorias de la siguiente fase.
- **Por qué importa la calidad:** las decisiones posteriores (minería de secuencias y transiciones
  por persona) son tan buenas como lo sea el dato de entrada; un nulo no explicado o un duplicado
  mal entendido cambia la trayectoria calculada para una persona.


## 2. Carga del dataset

**Archivo utilizado (localizado en el repositorio):**

| Atributo | Valor |
|---|---|
| Archivo | `data/cruce/empleos.parquet` |
| Formato | Parquet (columnar, versionado con git) |
| Por qué Parquet | conserva los tipos correctos y es el archivo acordado del proyecto; los CSV (antes `empleos.csv`, `empleos_ref.csv`) se eliminaron: no se versionan por tamaño (~164 MB) y se trabaja solo con parquet |
| Propósito | 1 fila = 1 experiencia laboral enriquecida (código + etiqueta + grupo ISCO) |
| Columnas | `resume_id, start_date, end_date, university_level, matched_code, emparejado, occupation_code, occupation_label, isco_group, isco_group_label, isco_level` |

El parquet es el **único** dataset integrado que existe físicamente y es el que referencia la
documentación de la rama. El código resuelve la ruta desde la raíz del proyecto, por lo que el
cuaderno funciona aunque se abra desde `libros/` o `script/`.


In [ ]:
# Configuración e importaciones del cuaderno: todo se declara una sola vez.
from pathlib import Path

import numpy as np
import pandas as pd
import unicodedata
from IPython.display import display

# Raíz del proyecto: se busca hacia arriba hasta encontrar la carpeta data/.
PROYECTO = Path.cwd()
while not (PROYECTO / "data").exists() and PROYECTO != PROYECTO.parent:
    PROYECTO = PROYECTO.parent

RUTA_EMPLEOS = PROYECTO / "data" / "cruce" / "empleos.parquet"
RUTA_EMPLEOS_LIMPIO_PARQUET = PROYECTO / "data" / "cruce" / "empleos_limpio.parquet"

# Configuración del diagnóstico (parámetros del negocio, un único punto).
ANIO_CORTE = 2026             # Año del proyecto: fechas posteriores son errores de captura.
ANIO_LIMITE_HISTORICO = 1990  # Ventana temporal histórica razonable (decisión a discutir).
SENTINELA_FIN = 99999         # 'Present' sin fecha de fin: se ordena al final en solapamientos.

print("Raíz del proyecto:", PROYECTO)
print("Dataset integrado :", RUTA_EMPLEOS)


**Explicación del código**

- Todas las importaciones y las rutas viven en **un solo bloque de configuración**: las celdas
  siguientes no declaran imports sueltos ni repiten caminos textuales.
- `ANIO_CORTE` y `ANIO_LIMITE_HISTORICO` son los únicos parámetros de negocio: cambiarlos aquí
  modifica todo el diagnóstico sin tocar el resto del código.
- No se ejecuta ninguna transformación todavía: solo se prepara el entorno.


In [ ]:
def cargarDatasetIntegrado(ruta: Path) -> pd.DataFrame:
    """Carga el dataset integrado de empleos desde un archivo Parquet.

    Se utiliza Parquet porque conserva los tipos reales (texto, entero) y es el
    archivo versionado del proyecto.
    """
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró el dataset integrado en la ruta {ruta}.\n"
            "Verifique que exista data/cruce/empleos.parquet antes de ejecutar."
        )
    return pd.read_parquet(ruta)

**Explicación del código**

- `cargarDatasetIntegrado` es la única puerta de entrada al dato: si el archivo no existe, falla
  con un mensaje claro en lugar de trabajar con un DataFrame vacío.
- Separar la carga en una función permite reutilizarla (también sirve para re-cargar el limpio
  si se quiere comparar) y mantiene una sola responsabilidad.


In [ ]:
empleos = cargarDatasetIntegrado(RUTA_EMPLEOS)

print("Filas:", len(empleos), "| Columnas:", empleos.shape[1])
empleos.head(3)

**Estructura esperada**

- El `.head()` muestra las primeras filas para confirmar visualmente las 11 columnas y su contenido.
- Según la auditoría previa de la rama, se esperan **1.506.445 filas × 11 columnas** y
  **284.247 personas** (`resume_id` únicos). Esos números se vuelven a calcular al ejecutar.


## Funciones auxiliares de fechas

Definir el modelo temporal **una sola vez** antes de los diagnósticos evita repetir la misma
lógica en cada sección (sección 3 para rangos, sección 5 para duraciones y solapamientos,
sección 7 para la estadística de la duración).


In [ ]:
def convertirTrimestreOrdinal(serie: pd.Series) -> pd.Series:
    """Convierte 'Q<n> <aaaa>' al ordinal trimestral (año*4 + trimestre).

    'Present' o cualquier valor no parseable devuelve NaN (no es fecha de fin).
    """
    partes = serie.str.extract(r"^Q([1-4])\s+(\d{4})$")
    anio = pd.to_numeric(partes[1], errors="coerce")
    trimestre = pd.to_numeric(partes[0], errors="coerce")
    return anio * 4 + trimestre

def extraerAnio(df: pd.DataFrame, columna: str) -> pd.Series:
    """Extrae el año (entero) de una columna con formato 'Qn AAAA'."""
    return pd.to_numeric(df[columna].str.extract(r"(\d{4})$", expand=False), errors="coerce")

def calcularDuracionTrimestres(df: pd.DataFrame) -> pd.Series:
    """Duración por fila en trimestres (inclusiva): ordinal(end) - ordinal(start) + 1.

    'Present' u otra fila sin fecha de fin devuelve NaN (censura a la derecha).
    """
    return convertirTrimestreOrdinal(df["end_date"]) - convertirTrimestreOrdinal(df["start_date"]) + 1


**Explicación del código**

- `convertirTrimestreOrdinal` transforma el texto a un número comparable: así `Q4 1999` vs
  `Q1 2000` se ordenan correctamente (de otra forma ordenaría por string y mezclaría años).
- `extraerAnio` aísla el año para diagnóstico de rangos y fechas futuras.
- `calcularDuracionTrimestres` define la duración en **un solo lugar**: las secciones 5 y 7 la
  reutilizan, por lo que no hay dos implementaciones de la misma regla.


## 3. Diagnóstico general

El principio de la limpieza es: **diagnosticar antes de tocar**. Cada diagnóstico tiene su
código y, inmediatamente después, la explicación de qué mide, qué problema busca, cómo se
interpreta y por qué importa.


In [ ]:
def diagnosticarDimensiones(df: pd.DataFrame) -> dict:
    """Devuelve las dimensiones del dataset (filas y columnas)."""
    return {"filas": len(df), "columnas": df.shape[1]}

def diagnosticarTipos(df: pd.DataFrame) -> pd.DataFrame:
    """Devuelve el tipo de dato de cada columna."""
    return df.dtypes.rename("tipo").to_frame()

def diagnosticarValoresNulos(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta valores nulos (NaN) por columna, en cantidad y porcentaje."""
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    porcentaje = (nulos / len(df) * 100).round(2)
    return pd.DataFrame({"cantidad": nulos, "%": porcentaje})

def contarValoresUnicos(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta valores únicos por columna (el NaN se cuenta si existe)."""
    return df.nunique(dropna=False).rename("valores_unicos").to_frame()

**Explicación del código**

- Cada función mide **una** cosa: dimensiones, tipos, nulos y cardinalidad. Eso las hace
  reutilizables y fáciles de interpretar (responsabilidad única, primer principio SOLID).
- `diagnosticarValoresNulos` filtra a las columnas con nulos reales: si una columna no tiene NaN,
  no aparece en la tabla (no se dibujan ceros que distraen).
- Los nombres siguen la convención del cuaderno: **camelCase en español** y verbo inicial.


In [ ]:
print("Dimensiones:", diagnosticarDimensiones(empleos))
print("\nTipos de datos:")
display(diagnosticarTipos(empleos))
print("\nValores nulos por columna:")
display(diagnosticarValoresNulos(empleos))
print("\nValores únicos por columna:")
display(contarValoresUnicos(empleos))

**Cómo interpretar (referencia previa de la auditoría)**

- **Nulos:** la integración dejó nulos solo donde el `matched_code` no pudo emparejarse con una
  ocupación: `occupation_code/label` ≈ 115.169 (7,64 %) y `isco_group/label/level` ≈ 104.993
  (6,97 %); el resto de columnas no tendría nulos (0 %). Son **nulos estructurales**, no pérdida
  de datos: se estudian en la sección 8.
- **Tipos:** texto salvo `resume_id` (entero) e `isco_level` (numérico, constante 4). No existe
  columna numérica "analítica" real; la única numérica con sentido es la duración, que se deriva.
- **Únicos:** se espera `matched_code` ≈ 2.983, `occupation_label` ≈ 2.966, `isco_group_label` ≈ 426
  y `university_level` = 5 categorías.


In [ ]:
def contarLiteralesDesconocidos(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta literales de datos ausentes/desconocidos y NaN estructurales.

    'unknown' y 'None' son cadenas del dataset original, no NaN: distinguirlos del
    NaN técnico es parte del diagnóstico de calidad.
    """
    checks = {
        "matched_code == 'unknown'": df["matched_code"].eq("unknown"),
        "university_level == 'None'": df["university_level"].eq("None"),
        "end_date == 'Present'": df["end_date"].eq("Present"),
        "occupation_code es NaN": df["occupation_code"].isna(),
        "isco_group es NaN": df["isco_group"].isna(),
    }
    cantidades = pd.Series({clave: int(mascara.sum()) for clave, mascara in checks.items()})
    return pd.DataFrame(
        {"cantidad": cantidades, "%": (cantidades / len(df) * 100).round(2)}
    )

display(contarLiteralesDesconocidos(empleos))

**Cómo interpretar (referencia previa)**

- `matched_code == 'unknown'` ≈ 104.993: empleos sin código clasificable desde la fuente.
- `university_level == 'None'` ≈ 174.036: personas que no reportaron formación (literal, no NaN).
- `end_date == 'Present'` ≈ 76.180: empleos vigentes (censura a la derecha).
- `occupation_code` NaN (≈ 115.169) debe ser **explicable** por `unknown` (104.993) + `rescatado`
  (10.176): eso se valida como regla de consistencia en la sección 6.


In [ ]:
def distribucionDe(df: pd.DataFrame, columna: str) -> pd.DataFrame:
    """Distribución absoluta y relativa de una columna categórica."""
    serie = df[columna].value_counts(dropna=False).rename("cantidad").to_frame()
    serie["porcentaje"] = (serie["cantidad"] / len(df) * 100).round(2)
    return serie

print("Distribución de emparejado:")
display(distribucionDe(empleos, "emparejado"))
print("\nDistribución de university_level:")
display(distribucionDe(empleos, "university_level"))

**Cómo interpretar (referencia previa)**

- `emparejado`: `ok` 1.391.276 (92,35 %) · `unknown` 104.993 (6,97 %) · `rescatado` 10.176 (0,68 %)
  · `descartado` debe ser **0** (nada quedó sin clasificar).
- `university_level`: 5 categorías — Secondary 590.624 · Bachelor 515.463 · Master 219.884 ·
  None 174.036 · PhD 6.438. La categoría "None" es el problema MNAR que se re-categoriza (§ 11).


In [ ]:
def analizarFechas(df: pd.DataFrame) -> pd.DataFrame:
    """Rango anual (mín/máx), 'Present' y no-fechas por variable temporal.

    El rango se calcula con el año extraído (no con el texto 'Qn AAAA'): ordenar
    el texto mezclaría años porque 'Q1 2000' < 'Q4 1999' lexicográficamente.
    """
    patron = r"^Q[1-4]\s+\d{4}$"
    filas = []
    for columna in ["start_date", "end_date"]:
        marcas = df[columna]
        formato_ok = marcas.str.match(patron, na=False)
        anios = extraerAnio(df, columna).dropna()
        filas.append({
            "columna": columna,
            "año_mínimo": int(anios.min()) if len(anios) else None,
            "año_máximo": int(anios.max()) if len(anios) else None,
            "con 'Present'": int(marcas.eq("Present").sum()),
            "fuera_de_formato": int((~formato_ok & marcas.notna() & marcas.ne("Present")).sum()),
        })
    return pd.DataFrame(filas)

display(analizarFechas(empleos))

**Cómo interpretar (referencia previa)**

- Rango esperado 1955–2029; `end_date` tiene ≈ 76.180 "Present" (no es fecha) y **0**
  valores fuera de formato `Q<n> <aaaa>`.
- El "2029" ya delata el problema de las fechas futuras que se diagnostica en la sección 5.


## 4. Diagnóstico de duplicados

**No se asume que duplicado = error.** Primero hay que decidir qué significa "duplicado" en este
dataset:

- Una fila es **duplicada real** si repite exactamente la misma experiencia: misma persona
  (`resume_id`), misma ocupación (`matched_code`), mismo trimestre de inicio y de fin.
- Eso define la llave natural: **PK = (`resume_id`, `matched_code`, `start_date`, `end_date`)**.
- Dos filas con el mismo inicio/fin pero **distinta ocupación** suelen ser **pluriempleo real**
  (dos trabajos a la vez), no un error. Eliminarlas borraría información válida.


In [ ]:
def detectarDuplicados(df: pd.DataFrame) -> int:
    """Número de filas duplicadas exactas (todas las columnas iguales)."""
    return int(df.duplicated().sum())

def contarGruposDuplicados(df: pd.DataFrame, columnas: list) -> int:
    """Filas extra tras conservar la primera ocurrencia de cada combinación (keep='first')."""
    return int(df.duplicated(subset=columnas, keep="first").sum())

CLAVE_PRIMARIA_EMPLEOS = ["resume_id", "matched_code", "start_date", "end_date"]

claves = [
    ("Exactas (fila completa)", detectarDuplicados(empleos)),
    ("Por PK natural (resume, code, ini, fin)", contarGruposDuplicados(empleos, CLAVE_PRIMARIA_EMPLEOS)),
    ("Por (resume_id, start_date, end_date)", contarGruposDuplicados(empleos, ["resume_id", "start_date", "end_date"])),
    ("Por (resume_id, start_date, matched_code)", contarGruposDuplicados(empleos, ["resume_id", "start_date", "matched_code"])),
]
display(pd.DataFrame(claves, columns=["tipo_de_duplicado", "filas"]))


**Cómo interpretar (referencia previa)**

- Exactas: **0**. Por PK natural: **0** → no existe la experiencia repetida idéntica.
- Las otras dos mediciones **no son duplicados**: son filas de personas con varios empleos en un
  mismo trimestre o con el mismo empleo re-registrado. La auditoría registró ≈ 74.357 (mismo
  inicio/fin) y ≈ 9.601 (mismo inicio/código).

**Decisión provisional:** sin duplicados por PK, no hay filas que eliminar por este motivo. Las
"claves cortas" se conservan y se interpretan como pluriempleo/transición (regla de negocio, no de
borrado). Si al ejecutar el diagnóstico aparecieran duplicados por PK, se revisaría fila por fila
antes de cualquier eliminación.


## 5. Diagnóstico de fechas

**Matemática de la duración.** Cada fecha tiene el formato `Q<n> <aaaa>`. Se convierte en un
ordinal trimestral:

    ordinal(año, trimestre) = año × 4 + trimestre

Por ejemplo, `Q1 2000` → 2000×4+1 = 8.001 y `Q3 2001` → 8.007. La duración se define como

    duración (trimestres) = ordinal(end_date) − ordinal(start_date)

De modo que una experiencia de un solo trimestre dura **0** trimestres. `Present` no tiene fecha
de fin: la duración queda indefinida (censura a la derecha, no un error).


In [ ]:
def contarInicioDespuesFin(df: pd.DataFrame) -> int:
    """Filas con start_date posterior a end_date (temporalmente imposibles)."""
    return int((calcularDuracionTrimestres(df) < 0).sum())

def contarDuracionCero(df: pd.DataFrame) -> int:
    """Experiencias del mismo trimestre de inicio y fin (duración 0)."""
    return int((calcularDuracionTrimestres(df) == 0).sum())

def contarFechasFuturas(df: pd.DataFrame, anioCorte: int) -> int:
    """Filas con alguna fecha cuyo año supera el corte (año del proyecto)."""
    anio_inicio = extraerAnio(df, "start_date")
    anio_fin = extraerAnio(df, "end_date")
    return int(((anio_inicio > anioCorte) | (anio_fin > anioCorte)).sum())

def contarIniciosAntiguos(df: pd.DataFrame, anioLimite: int) -> int:
    """Inicios con año anterior o igual al límite (cola histórica)."""
    return int((extraerAnio(df, "start_date") <= anioLimite).sum())

def contarFilasSolapadas(df: pd.DataFrame) -> int:
    """Filas de una misma persona cuyo empleo se solapa con el anterior.

    'Present' se interpreta como vigente hasta SENTINELA_FIN (fin de la numeración), por lo que un
    empleo sin fin se solapa con todos los posteriores de la persona.
    """
    trabajo = df[["resume_id", "start_date", "end_date"]].copy()
    trabajo["_ini"] = convertirTrimestreOrdinal(trabajo["start_date"])
    trabajo["_fin"] = convertirTrimestreOrdinal(trabajo["end_date"]).fillna(SENTINELA_FIN)
    trabajo = trabajo.sort_values(["resume_id", "_ini", "_fin"])
    fin_anterior = trabajo.groupby("resume_id")["_fin"].shift(1)
    return int((trabajo["_ini"] <= fin_anterior).sum())


**Explicación del código**

- Todas las funciones reutilizan los auxiliares `calcularDuracionTrimestres` y `extraerAnio`:
  no existe lógica de fechas duplicada.
- `contarFilasSolapadas` ordena cada persona por inicio y compara cada fila con la **anterior**
  (`groupby().shift()`): si el inicio actual es anterior o igual al fin de la anterior, hay
  solapamiento. Todo es vectorizado, sin ciclos fila por fila.


In [ ]:
fechas = [
    ("Nulos en start_date", int(empleos["start_date"].isna().sum())),
    ("Nulos en end_date", int(empleos["end_date"].isna().sum())),
    ("end_date == 'Present' (vigente)", int(empleos["end_date"].eq("Present").sum())),
    ("start > end", contarInicioDespuesFin(empleos)),
    ("Duración 0 (mismo trimestre)", contarDuracionCero(empleos)),
    (f"Fechas futuras (año > {ANIO_CORTE})", contarFechasFuturas(empleos, ANIO_CORTE)),
    (f"Inicios ≤ {ANIO_LIMITE_HISTORICO}", contarIniciosAntiguos(empleos, ANIO_LIMITE_HISTORICO)),
    ("Filas solapadas (misma persona)", contarFilasSolapadas(empleos)),
]
display(pd.DataFrame(fechas, columns=["diagnóstico", "cantidad"]))

**Cómo interpretar (referencia previa)**

- Nulos en fechas: **0** · `start > end`: **0** · duración 0: existe y se conserva (empleo de un
  solo trimestre es válido).
- **Fechas futuras:** ≈ **11** filas (>2026: 1 inicio + 11 fin) → error de captura, se tratan en la
  limpieza.
- **Inicios ≤ 1990:** ≈ 75.227 → cola histórica; se documenta, no se borra (decisión de negocio).
- **Solapamientos:** ≈ 429.958 filas · 154.131 personas → pluriempleo/transición real, se conserva.


## 6. Diagnóstico de consistencia entre columnas

Se validan **reglas reales** que la integración debe haber respetado. Cada regla produce un conteo
de violaciones; el esperado es 0 salvo las cadenas de nulos estructurales, ya explicadas.


In [ ]:
def validarConsistencia(df: pd.DataFrame) -> pd.DataFrame:
    """Cuenta violaciones de las reglas de consistencia interna del integrado."""
    reglas = [
        (
            "emparejado='unknown' ⟺ occupation_code NaN",
            int(np.logical_xor(df["emparejado"].eq("unknown"), df["occupation_code"].isna()).sum()),
        ),
        (
            "emparejado='rescatado' ⟹ occupation NaN y isco presente",
            int(((df["emparejado"].eq("rescatado"))
                 & ~(df["occupation_code"].isna() & df["isco_group"].notna())).sum()),
        ),
        (
            "emparejado='ok' ⟹ occupation e isco presentes",
            int(((df["emparejado"].eq("ok"))
                 & (df["occupation_code"].isna() | df["isco_group"].isna())).sum()),
        ),
        (
            "isco_group presente ⟹ isco_level == 4",
            int(((df["isco_group"].notna()) & (df["isco_level"] != 4)).sum()),
        ),
        (
            "isco_group NaN ⟺ isco_group_label NaN",
            int(np.logical_xor(df["isco_group"].isna(), df["isco_group_label"].isna()).sum()),
        ),
        (
            "occupation_code presente ⟺ occupation_label presente",
            int(np.logical_xor(df["occupation_code"].notna(), df["occupation_label"].notna()).sum()),
        ),
    ]
    return pd.DataFrame(reglas, columns=["regla", "violaciones"])

display(validarConsistencia(empleos))

**Explicación del código**

- Se usan operaciones vectorizadas (`eq`, `isna`, `np.logical_xor`) sobre columnas completas.
- `logical_xor` detecta que ambas condiciones coincidan: p. ej., "emparejado unknown" debe ser
  exactamente igual a "occupation NaN". Si una fila tuviera `unknown` con ocupación, sería error.
- La función se reutiliza después de la limpieza (sección 11) para confirmar que ninguna regla se
  rompió.

**Resultado esperado:** 0 violaciones. Si alguna regla reportara > 0, se investigaría antes de
limpiar: sería un defecto de la integración, no un caso que la limpieza deba "arreglar" a ciegas.


## 7. Diagnóstico estadístico y valores atípicos

La única variable numérica con sentido real es la **duración en trimestres** (`dur_Q`
= `calcularDuracionTrimestres`), porque `isco_level` es una constante y el resto son
códigos/textos. La estadística se aplica donde tiene sentido, no "porque las técnicas existen".


In [ ]:
def calcularLimitesOutliers(vals: pd.Series) -> dict:
    """Límites IQR (regla de Tukey): [Q1 − 1.5·IQR, Q3 + 1.5·IQR].

    Es la regla del curso para datos con colas largas: robusta frente a extremos.
    """
    q1, q3 = vals.quantile([0.25, 0.75])
    iqr = q3 - q1
    return {"q1": q1, "q3": q3, "iqr": iqr,
            "inferior": q1 - 1.5 * iqr, "superior": q3 + 1.5 * iqr}

def calcularZScores(vals: pd.Series) -> pd.Series:
    """z-score = (x − media) / desviación estándar. Solo como comparación."""
    return (vals - vals.mean()) / vals.std()

**Explicación del código**

- `calcularLimitesOutliers` implementa la regla Tukey que se usa como criterio principal.
- `calcularZScores` se conserva solo para comparar (no como criterio), porque la media se
  arrastra por la cola y "perdona" valores que el IQR sí marca.
- La duración proviene de la función auxiliar única, no de una copia local.


In [ ]:
dur_valida = calcularDuracionTrimestres(empleos).dropna()

print("Duraciones computadas:", len(dur_valida))
print("\nResumen numérico de la duración (trimestres):")
display(dur_valida.describe().rename("dur_Q"))

percentiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
display(dur_valida.quantile(percentiles).to_frame("dur_Q").rename(index={p: f"{int(p*100)}%" for p in percentiles}))

limites = calcularLimitesOutliers(dur_valida)
print("\nLímites IQR (Tukey):", limites)

fuera = (dur_valida < limites["inferior"]) | (dur_valida > limites["superior"])
personas_fuera = empleos.loc[dur_valida[fuera].index, "resume_id"].nunique()
print("Valores fuera de límites:", int(fuera.sum()), "| personas:", personas_fuera)

z = calcularZScores(dur_valida)
print("|z| > 3 (comparación):", int((z.abs() > 3).sum()))

**Cómo interpretar (referencia previa)**

- Duración: mediana **5**, P90 **24**, P95 **36**, P99 **73**, máximo **160** trimestres
  (≈40 años → carrera real). IQR = 9 (Q1=2, Q3=11), límite superior ≈ 24,5.
- Fuera de límites ≈ **141.591** filas / **97.954** personas, **todas por arriba** (empleos ≥ 25
  trimestres). El |z|>3 marca ≈ 33.841: la disparidad entre ambos métodos confirma la cola larga.

**Decisión:** la cola larga es **señal de estabilidad laboral**, no error. No se elimina ni se
trunca; solo se contemplaría winsorizar si `dur_Q` entrara a un modelo, y siempre sobre el
original. Las únicas "anomalías puntuales" reales son las 12 fechas futuras (error de dominio).


## 8. Exactitud, precisión y consistencia

Este apartado deja **explícito qué se puede y qué no se puede verificar** con los datos disponibles.
No se inventa una "exactitud del X %": sin una fuente de verdad externa no es medible.

| Aspecto | Qué se puede verificar aquí |
|---|---|
| **Exactitud verificable** | Nada semántico: sin fuente externa no se puede confirmar que un empleo "realmente" fue de tal ocupación. Solo se verifica coherencia interna. |
| **Precisión / formato verificable** | Formato `Q<n> <aaaa>` en fechas, tipos correctos, etiquetas sin duplicados textuales tras normalizar, `isco_level` constante. |
| **Consistencia interna** | Las reglas de la sección 6 (emparejado ↔ nulos, label ↔ código, ISCO ↔ nivel) y que cada categoría se escriba siempre igual. |
| **No verificable** | Exactitud del `matched_code` original, la veracidad de `university_level`, la validez del `resume_id`. Se asumen como fuente (job = dato primario). |

**Clasificación de ausencias (MCAR / MAR / MNAR), para elegir el método correcto:**

- **MCAR** (azar): las 12 fechas futuras → corregir/eliminar es legítimo.
- **MAR estructural:** el NaN de ocupación se explica 100 % por `emparejado` → bandera, no imputar.
- **MNAR:** `university_level='None'` (quien no reportó) → re-categorizar como categoría propia,
  no imputar moda.


In [ ]:
def normalizarTexto(serie: pd.Series) -> pd.Series:
    """Minúsculas, sin espacios extremos ni tildes (para comparar etiquetas)."""
    normalizada = serie.str.strip().str.lower()
    return (
        normalizada.str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
    )

def contarColisionesEtiquetas(serie: pd.Series) -> dict:
    """Etiquetas que colisionan tras normalizar (strip + minúsculas + sin tildes).

    Una colisión implica que la misma etiqueta aparece escrita de dos maneras,
    lo que rompería agrupaciones futuras.
    """
    originales = serie.nunique(dropna=False)
    normalizadas = normalizarTexto(serie.dropna()).nunique()
    return {"etiquetas_originales": originales,
            "etiquetas_normalizadas": normalizadas,
            "posibles_colisiones": originales - normalizadas}

for columna in ["occupation_label", "isco_group_label", "emparejado", "university_level"]:
    print(columna, "->", contarColisionesEtiquetas(empleos[columna]))

**Explicación del código**

- `normalizarTexto` es una función auxiliar reutilizable (espacios, minúsculas, tildes) y
  `contarColisionesEtiquetas` la usa para comparar cardinalidades: detecta "lo mismo escrito de
  dos formas", que es el riesgo clásico de precisión.
- Se esperan **0 colisiones** (la integración ya normalizó); el chequeo se conserva como
  re-verificación defensiva, no como transformación.

**Conclusión de calidad:** el dataset es internamente consistente; los defectos relevantes son de
**dominio** (fechas futuras), **semánticos** (`None`, `Present`) y **estructurales** (nulos de
ocupación explicados por banderas). Exactitud externa: **no verificable**, y así se declara.


## 9. Resumen de problemas encontrados

La tabla describe los problemas que el diagnóstico **busca confirmar** al ejecutar el cuaderno
(no resultados inventados). La columna "Evidencia" es el dato que el código de las secciones 3–8
va a imprimir.

| Problema | Evidencia (a confirmar al ejecutar) | Impacto | Acción propuesta |
|---|---|---|---|
| Ocupación/área nulas | `occupation_code` NaN ≈ 115.169 (7,64 %); `isco_*` NaN ≈ 104.993 | 115.169 filas sin nombre de ocupación; 104.993 sin área | Bandera `es_unknown_ocupacion` / `es_rescatado`; conservar |
| `matched_code == 'unknown'` | ≈ 104.993 (6,97 %) | Empleos no clasificables desde la fuente | Bandera; no imputar |
| `university_level == 'None'` | ≈ 174.036 (11,55 %) | Categoría ambigua; imputar moda falsearía datos | Re-categorizar a "No reportado" |
| `end_date == 'Present'` | ≈ 76.180 (5,06 %) | Duración indefinida (censura) | Bandera `es_vigente` |
| Duplicados por claves cortas | (resume, start, end) ≈ 74.357; (resume, start, code) ≈ 9.601 | Riesgo de borrar pluriempleo válido | No eliminar; documentar como transición |
| Fechas futuras | ≈ 11 filas (> 2026) | Error de captura, imposible | Eliminar (MCAR, < 1 %) |
| Cola larga de duración | ≈ 141.591 duraciones fuera del IQR | Carreras largas legítimas (~40 años) | Conservar; winsorizar solo a demanda |
| Ventana temporal histórica | ≈ 75.227 inicios ≤ 1990 | Datos muy antiguos | Documentar, no borrar (decisión de negocio) |


## 10. Reglas de limpieza (diagnóstico → decisión → método)

| Problema | Método | Justificación | Resultado esperado |
|---|---|---|---|
| Duplicados (por PK) | Sin acción si 0; revisión manual si > 0 | La experiencia repetida idéntica es el único duplicado real | 0 filas eliminadas por duplicados |
| Ocupación/área sin emparejar | Bandera `es_unknown_ocupacion`/`es_rescatado` | El NaN es MAR estructural: "no emparejado" ≠ "sin dato"; imputar inventa | NaN explicados por banderas |
| `university_level='None'` | Re-categorizar a "No reportado" | MNAR; la moda inflaría "Secondary school" | Categoría propia, sin `None` |
| `end_date='Present'` | Bandera `es_vigente` | Censura real, no fecha inventada | Duración indefinida marcada |
| Fechas futuras | Eliminar filas | Error de dominio, MCAR, < 1 % | 0 fechas futuras |
| Cola larga `dur_Q` | Conservar (sin winsorizar por ahora) | Estabilidad = señal; solo winsorizar si entra a modelo, sobre el original | 0 filas eliminadas |
| Exactitud externa | No aplicar | No hay fuente de verdad | Se declara no verificable |


In [ ]:
# Bitácora: registro estructurado de cada decisión, para trazabilidad.
BITACORA = []

def anotarBitacora(bitacora: list, columna: str, problema: str, cantidad: int,
                   metodo: str, razon: str, impacto: str) -> None:
    """Agrega una fila a la bitácora de la limpieza (el original nunca se modifica)."""
    bitacora.append({
        "columna": columna,
        "problema": problema,
        "cantidad": cantidad,
        "metodo": metodo,
        "razon": razon,
        "impacto": impacto,
    })

**Explicación del código**

- `BITACORA` acumula, fila por fila, cada transformación: qué columna, qué problema, cuántas filas,
  qué método, por qué y qué impacto tiene. Al final se imprime como tabla (trazabilidad completa
  original → limpio).


## 11. Implementación de la limpieza

Cada bloque sigue la secuencia **Problema → Código → Explicación → Justificación → Validación**.
La limpieza se aplica siempre sobre una **copia de trabajo**; el dataset original no se toca.


In [ ]:
def crearCopiaTrabajo(df: pd.DataFrame) -> pd.DataFrame:
    """Copia de trabajo: el dataset original nunca se modifica."""
    return df.copy()

df_limpio = crearCopiaTrabajo(empleos)
print("Copia de trabajo creada:", df_limpio.shape)

**Problema:** el dataset original es el registro de partida de la trazabilidad.
**Explicación del código:** `crearCopiaTrabajo` entrega una copia; toda transformación posterior
actúa sobre `df_limpio`.
**Justificación:** si una celda limpia mal, el original queda intacto y se puede reiniciar.
**Validación:** `empleos.shape == df_limpio.shape` al inicio (se imprime arriba).


In [ ]:
def validarSinDuplicados(df: pd.DataFrame, clave: list) -> bool:
    """Valida que no existan duplicados exactos ni por llave natural."""
    assert df.duplicated().sum() == 0, "Existen duplicados exactos"
    assert df.duplicated(subset=clave).sum() == 0, "Existen duplicados por llave natural"
    return True

validarSinDuplicados(df_limpio, CLAVE_PRIMARIA_EMPLEOS)
print("Duplicados exactos y por llave natural: 0  →  sin acción necesaria.")

**Problema que mitiga:** ninguno, en realidad: se despeja el diagnóstico. **Por qué no se eliminan
las claves cortas:** (resume, start, end) y (resume, start, code) representan pluriempleo o
transición real; borrarlas eliminaría trayectorias válidas (≈74.357 y ≈9.601 filas).
**Validación:** los `assert` fallan ruidosamente si apareciera un duplicado real, obligando a
revisar antes de continuar.


In [ ]:
def agregarBanderasEmparejamiento(df: pd.DataFrame) -> pd.DataFrame:
    """Banderas de ocupación: 'unknown' y 'rescatado' (sin re-imputar)."""
    copia = df.copy()
    copia["es_unknown_ocupacion"] = copia["emparejado"].eq("unknown")
    copia["es_rescatado"] = copia["emparejado"].eq("rescatado")
    return copia

def agregarBanderaVigente(df: pd.DataFrame) -> pd.DataFrame:
    """Bandera de empleo vigente: end_date == 'Present' (censura)."""
    copia = df.copy()
    copia["es_vigente"] = copia["end_date"].eq("Present")
    return copia

def validarBanderas(df: pd.DataFrame) -> pd.DataFrame:
    """Comprueba que cada bandera coincide con su variable de origen."""
    return pd.DataFrame({
        "bandera": ["es_unknown_ocupacion", "es_rescatado", "es_vigente"],
        "activos": [
            int(df["es_unknown_ocupacion"].sum()),
            int(df["es_rescatado"].sum()),
            int(df["es_vigente"].sum()),
        ],
        "esperado": [
            int(df["emparejado"].eq("unknown").sum()),
            int(df["emparejado"].eq("rescatado").sum()),
            int(df["end_date"].eq("Present").sum()),
        ],
    })

df_limpio = agregarBanderasEmparejamiento(df_limpio)
df_limpio = agregarBanderaVigente(df_limpio)
print("Nuevas columnas:", [c for c in df_limpio.columns if c.startswith("es_")])

val_banderas = validarBanderas(df_limpio)
display(val_banderas)
assert (val_banderas["activos"] == val_banderas["esperado"]).all()

anotarBitacora(
    BITACORA, "occupation_code/isco_*",
    "NaN estructural (emparejado unknown/rescatado)",
    int(df_limpio["occupation_code"].isna().sum()),
    "Banderas es_unknown_ocupacion y es_rescatado",
    "El NaN informa 'sin emparejar', no la ausencia de dato; imputar inventaría ocupaciones.",
    "Se conservan las filas; el NaN queda explicado.",
)

**Problema:** nulos estructurales de ocupación/área (MAR).
**Explicación del código:** `agregarBanderasEmparejamiento` y `agregarBanderaVigente` derivan las
banderas booleanas de columnas ya clasificadas; `validarBanderas` las cruza contra su origen y el
`assert` garantiza la coincidencia. La decisión se registra en la bitácora.
**Justificación:** la ausencia significa algo ("no emparejado" o "vigente") y ningún método
imputaría bien un código ocupacional; una bandera conserva toda la información sin fabricar datos.
**Validación:** banderas = conteos de origen (tabla + `assert`).


In [ ]:
def limpiarNivelEducativo(df: pd.DataFrame) -> pd.DataFrame:
    """'None' de university_level pasa a ser la categoría explícita 'No reportado'."""
    copia = df.copy()
    copia["university_level"] = copia["university_level"].replace({"None": "No reportado"})
    return copia

df_limpio = limpiarNivelEducativo(df_limpio)

categorias = df_limpio["university_level"].value_counts(dropna=False).rename_axis(None)
print("Categorías de university_level tras la transformación:")
print(categorias.to_string())
assert "None" not in set(df_limpio["university_level"].dropna())

anotarBitacora(
    BITACORA, "university_level",
    "Literal 'None' (MNAR)",
    int(empleos["university_level"].eq("None").sum()),
    "Re-categorización a 'No reportado'",
    "Imputar la moda (Secondary school) fabricaría educación para 174.036 filas.",
    "Categoría propia y sin ambigüedad.",
)

**Problema:** `university_level == 'None'` es un literal que parece dato pero dice "sin dato"
(MNAR: quien no lo reportó).
**Explicación del código:** `.replace({"None": "No reportado"})` re-etiqueta el literal; el `assert`
garantiza que no queda ningún `None` restante.
**Justificación:** re-categorizar es honesto: no inventa nivel educativo y deja la categoría
explícita para el análisis.
**Validación:** la impresión de `value_counts` y el `assert` (esperado: "No reportado" ≈ 174.036).


In [ ]:
def eliminarFechasFuturas(df: pd.DataFrame, anioCorte: int) -> tuple:
    """Elimina filas con fechas posteriores al corte (error de captura documentado).

    Solo legítimo porque es < 1 % del dataset y aparece de forma aislada (MCAR):
    no hay manera de corregir el trimestre real de esas filas.
    """
    anio_inicio = extraerAnio(df, "start_date")
    anio_fin = extraerAnio(df, "end_date")
    futura = (anio_inicio > anioCorte) | (anio_fin > anioCorte)
    return df.loc[~futura].copy(), int(futura.sum())

df_limpio, n_futuras = eliminarFechasFuturas(df_limpio, ANIO_CORTE)
print("Filas con fechas futuras eliminadas:", n_futuras)
print("Shape tras la eliminación:", df_limpio.shape)
assert contarFechasFuturas(df_limpio, ANIO_CORTE) == 0

anotarBitacora(
    BITACORA, "start_date / end_date",
    f"Fechas con año > {ANIO_CORTE} (captura futura)",
    n_futuras,
    "Eliminación de filas (MCAR)",
    "Fecha posterior al año del proyecto es imposible; <1% del dato y aislado.",
    f"{n_futuras} filas eliminadas.",
)

**Problema:** ≈ 11 filas con fechas posteriores a 2026 (error de captura).
**Explicación del código:** `eliminarFechasFuturas` reutiliza `extraerAnio`, marca la fila si el
inicio o el fin la exceden y la remueve; devuelve el DataFrame limpio **y** la cantidad eliminada
para la bitácora.
**Justificación:** es el único caso aceptable de eliminación: MCAR, < 1 %, sin posible corrección.
**Validación:** `assert contarFechasFuturas(...) == 0` y registro en la bitácora.


In [ ]:
# Cola larga: se conserva a propósito. Chequeo de que ninguna decisión la alteró.
duraciones_finales = calcularDuracionTrimestres(df_limpio).dropna()
print("Duraciones computadas en el limpio:", len(duraciones_finales))
print("Filas del limpio (no se eliminó nada por outliers):", df_limpio.shape[0])

anotarBitacora(
    BITACORA, "dur_Q (derivada)",
    "Cola larga: duraciones ≥ 25 trimestres fuera del IQR",
    len(duraciones_finales),
    "Conservar (sin winsorizar por ahora)",
    "Carreras largas legítimas (~40 años); señal de estabilidad. Winsorizar solo si dur_Q entra a un modelo, sobre el original.",
    "0 filas eliminadas por atípicos.",
)

**Problema:** duraciones "extremas" según IQR (≈141.591 filas). **Explicación:** el código solo
recalcula y documenta; no trunca. **Justificación:** la cola es señal de empleos estables reales,
no medición errónea; borrar segmentaría personas completas. **Validación:** el shape no debe
cambiar por esta etapa (solo cambió por las fechas futuras de la etapa anterior).


In [ ]:
print("Reglas de consistencia sobre el limpio:")
display(validarConsistencia(df_limpio))

def contarNulosNoExplicados(df: pd.DataFrame) -> int:
    """Nulos en columnas sin bandera que los explique (deben ser 0)."""
    sin_bandera = ["resume_id", "start_date", "end_date", "university_level", "matched_code", "emparejado"]
    return int(df[sin_bandera].isna().sum().sum())

def contarNulosOcupacionNoExplicados(df: pd.DataFrame) -> int:
    """Nulos de ocupación que ninguna bandera explica (deben ser 0)."""
    faltante = df["occupation_code"].isna()
    explicado = faltante & (df["es_unknown_ocupacion"] | df["es_rescatado"])
    return int((faltante & ~explicado).sum())

print("Nulos sin bandera que los explique:", contarNulosNoExplicados(df_limpio))
print("Nulos de ocupación no explicados :", contarNulosOcupacionNoExplicados(df_limpio))

**Explicación del código:** re-ejecuta las reglas internas y verifica que todos los nulos
restantes estén **cubiertos por banderas** (nada queda sin explicación).
**Resultado esperado:** 0 violaciones y 0 nulos "huérfanos" — la "limpieza" no oculta datos, los
explica.


## 12. Validación final (post-limpieza)

Se re-ejecutan los diagnósticos clave sobre el limpio y se comparan contra el esperado.


In [ ]:
def validarResultadoLimpieza(limpio: pd.DataFrame) -> pd.DataFrame:
    """Re-ejecuta los chequeos clave y los compara contra su valor esperado (0)."""
    checks = {
        "nulos sin bandera que los explique": contarNulosNoExplicados(limpio),
        "nulos de ocupación no explicados por banderas": contarNulosOcupacionNoExplicados(limpio),
        "duplicados exactos": detectarDuplicados(limpio),
        "duplicados por llave natural": contarGruposDuplicados(limpio, CLAVE_PRIMARIA_EMPLEOS),
        "start > end": contarInicioDespuesFin(limpio),
        "fechas futuras": contarFechasFuturas(limpio, ANIO_CORTE),
        "violaciones de consistencia interna": int(validarConsistencia(limpio)["violaciones"].sum()),
    }
    resultado = pd.DataFrame(checks.items(), columns=["chequeo", "valor"])
    resultado["esperado"] = 0
    resultado["ok"] = resultado["valor"] == resultado["esperado"]
    return resultado

def construirTrazabilidad(original: pd.DataFrame, limpio: pd.DataFrame) -> pd.DataFrame:
    """Original vs limpio: filas y columnas, para la trazabilidad."""
    return pd.DataFrame([
        {"métrica": "filas", "original": len(original), "limpio": len(limpio),
         "diferencia": len(limpio) - len(original)},
        {"métrica": "columnas", "original": original.shape[1], "limpio": limpio.shape[1],
         "diferencia": limpio.shape[1] - original.shape[1]},
    ])

validacion_final = validarResultadoLimpieza(df_limpio)
display(validacion_final)
print("Todos los chequeos OK:", bool(validacion_final["ok"].all()))

print("\nTrazabilidad original → limpio:")
display(construirTrazabilidad(empleos, df_limpio))

print("\nBitácora de la limpieza:")
display(pd.DataFrame(BITACORA))

**Explicación del código**

- `validarResultadoLimpieza` vuelve a ejecutar los contadores y marca `ok` cuando coinciden con 0:
  la limpieza es **verificable**, no una promesa.
- `construirTrazabilidad` cuantifica filas/columnas iniciales vs finales (esperado: −11 filas por
  las fechas futuras; +3 columnas por las banderas).
- La bitácora queda impresa como tabla para el informe.
- La validación reutiliza las mismas funciones del diagnóstico: no hay "segunda implementación"
  del chequeo.


## 13. Dataset limpio

| Atributo | Valor |
|---|---|
| Nombre | `empleos_limpio.parquet` (única salida; los CSV no se versionan) |
| Ubicación | `data/cruce/` (misma carpeta que el integrado original, sin sobrescribirlo) |
| Formato | Parquet (columnar, versionado con git) |
| Columnas | las 11 originales + `es_unknown_ocupacion`, `es_rescatado`, `es_vigente` |
| Propósito | base verificada y documentada para la minería de trayectorias de la siguiente fase |

El archivo original `data/cruce/empleos.parquet` **no se modifica**; la exportación genera un
dataset derivado nuevo. La celda de exportación queda **comentada** para que el equipo decida
cuándo escribir el archivo (el notebook debe ejecutarse primero).


In [ ]:
# Exportación preparada, se activa cuando el equipo lo decida:
#
# df_limpio.to_parquet(RUTA_EMPLEOS_LIMPIO_PARQUET, index=False)
#
# Verificación posterior a la exportación:
# pd.read_parquet(RUTA_EMPLEOS_LIMPIO_PARQUET).shape  # debe coincidir con df_limpio.shape

## Notas finales

- Este cuaderno **no se ejecutó** durante su construcción: los números impresos se generan al
  correrlo (Kernel → Restart & Run All). Los valores de referencia citados provienen de la
  auditoría verificada de la rama (`md/AUDITORIA_RAMA_PRUEBA.md`) y sirven para interpretar el
  resultado esperado, no para reemplazarlo.
- No se modificó el dataset original ni se inventaron resultados.
- Estructura seguida: **diagnóstico → decisión → limpieza → validación → exportación**.
